1. Cargar el data set de la estructura de carpetas del proyecto

In [2]:
import pandas as pd
import os


In [ ]:
from sqlalchemy import create_engine, text


engine = create_engine(
    "mysql+pymysql://root:@127.0.0.1:3306/wdi_etl_db?charset=utf8mb4"
)

print("Conexión creada correctamente")

Conexión creada correctamente


#2.  Data Migration – Carga del Dataset a MySQL

En esta sección se realiza la fase de **Data Migration** del proceso ETL.  
El objetivo es cargar el dataset reducido del World Development Indicators (WDI) en una base de datos relacional (MySQL), cumpliendo con el requerimiento del proyecto.

Primero, se lee el archivo `WDI_10k_AGL.csv`, el cual contiene 10.000 registros y 69 columnas (features). Posteriormente, los datos se migran a la base de datos `wdi_etl_db`, creando la tabla `wdi_raw`. 

Finalmente, se ejecuta una consulta SQL para verificar que la carga se haya realizado correctamente, confirmando el número total de registros insertados.

Este paso garantiza que los datos estén almacenados en un sistema de gestión de bases de datos relacional, permitiendo su posterior transformación, análisis y consulta mediante SQL.

In [3]:


# Leer dataset
df = pd.read_csv("../data/raw/WDI_10k_AGL.csv")

print("Dimensiones del dataset:", df.shape)

# Cargar a MySQL
df.to_sql("wdi_raw", engine, if_exists="replace", index=False)

print(" Tabla 'wdi_raw' creada correctamente en MySQL")

# Verificar desde SQL
resultado = pd.read_sql("SELECT COUNT(*) AS total_filas FROM wdi_raw;", engine)
print(resultado)

Dimensiones del dataset: (10000, 69)
 Tabla 'wdi_raw' creada correctamente en MySQL
   total_filas
0        10000


#3. TRANSFORMACIÓN

##  Transform – Normalización de la tabla (formato Wide a Long)

El dataset WDI original está en formato **wide**, donde cada año (por ejemplo 1960, 1961, ..., 2022) aparece como una columna diferente.  
Para facilitar el análisis, consultas SQL y visualizaciones, se transforma la tabla a formato **long o tidy**, donde cada registro representa un valor único por:

- País  
- Indicador  
- Año  
- Valor  

El resultado se guarda como una nueva tabla llamada `wdi_clean`, que se considera la versión limpia y estructurada para el análisis posterior.

In [4]:


# 1) Identificar columnas que son años (por ejemplo "1960", "1961", ..., "2022")
year_cols = [c for c in df.columns if c.isdigit()]

print("Años detectados:", len(year_cols), "->", year_cols[:10], "...", year_cols[-5:])

# 2) Transformar a formato long
df_long = df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

# 3) Tipos correctos
df_long["Year"] = df_long["Year"].astype(int)
df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")

print("Dimensiones tabla long:", df_long.shape)
df_long.head()

Años detectados: 65 -> ['1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969'] ... ['2020', '2021', '2022', '2023', '2024']
Dimensiones tabla long: (650000, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,1960,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,1960,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,1960,NaN
3,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,1960,NaN
4,Africa Eastern and Southern,AFE,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,1960,NaN
